# Health Data Science — Mock training and Docker

Train a tiny decision tree, then use Docker to print a **confusion matrix**.
The ECG features, ages and labels are invented: this demonstrates deployment, not medical performance.

Save this notebook inside your local `hds-docker-practice` repository. Run the **three Python cells** in order, then use PowerShell for the commands below. Keep Docker Desktop running.

If packages are missing, run `%pip install pandas scikit-learn joblib` in a new cell, restart the kernel and start again.
The notebook creates or replaces the example files inside `docker_project`.

## 1. Train and save

We train on six rows and evaluate on six **different** mock rows. The test labels are saved with the test inputs so Docker can compare predictions with the expected answers.


In [1]:
from pathlib import Path
import pandas as pd
import joblib
from sklearn.tree import DecisionTreeClassifier

# Training examples: two features and a class label.
X_train = pd.DataFrame({
    "ecg_spread": [0.3, 0.4, 0.5, 0.8, 0.9, 1.0],
    "age": [25, 35, 45, 60, 70, 75],
})
y_train = [0, 0, 0, 1, 1, 1]

model = DecisionTreeClassifier(max_depth=1, random_state=42)
model.fit(X_train, y_train)

# Separate mock test examples, including some deliberately difficult cases.
test = pd.DataFrame({
    "ecg_spread": [0.35, 0.45, 0.55, 0.85, 0.95, 1.05],
    "age": [28, 38, 48, 63, 73, 78],
    "label": [0, 0, 1, 0, 1, 1],
})

folder = Path("docker_project")
folder.mkdir(exist_ok=True)
joblib.dump(model, folder / "model.joblib")
test.to_csv(folder / "test.csv", index=False)
print("Saved model.joblib and test.csv in", folder.resolve())


Saved model.joblib and test.csv in docker_project


## 2. Create the evaluation script

This script loads the saved model, predicts the test labels and prints the confusion matrix.
**Rows are actual classes; columns are predicted classes.** The diagonal contains correct predictions.
The `label` column is the answer, so it must not be an input feature.


In [2]:
script = """import joblib
import pandas as pd
from sklearn.metrics import confusion_matrix

model = joblib.load("model.joblib")
test = pd.read_csv("test.csv")

X_test = test[["ecg_spread", "age"]]
y_test = test["label"]
predictions = model.predict(X_test)

matrix = confusion_matrix(y_test, predictions, labels=[0, 1])
table = pd.DataFrame(matrix,
                     index=["Actual 0", "Actual 1"],
                     columns=["Predicted 0", "Predicted 1"])
print("Confusion matrix (mock test data):")
print(table)
"""

(folder / "run_project.py").write_text(script, encoding="utf-8")

# Run the same script locally before trying Docker.
import subprocess
import sys

result = subprocess.run([sys.executable, "run_project.py"], cwd=folder,
                        capture_output=True, text=True, check=True)
print(result.stdout)


Confusion matrix (mock test data):
          Predicted 0  Predicted 1
Actual 0            2            1
Actual 1            1            2



## 3. Create the Docker files

- **requirements.txt** lists the packages and their installed versions (`==` means exactly that version).
- **Dockerfile** starts with Python, copies our files, installs the packages and chooses the script to run.

Run this cell as supplied. No files need to be typed by hand.


In [3]:
from importlib.metadata import version

packages = ["pandas", "scikit-learn", "numpy", "scipy", "joblib", "threadpoolctl"]
requirements = "\n".join(name + "==" + version(name) for name in packages)
(folder / "requirements.txt").write_text(requirements + "\n", encoding="utf-8")

python_version = f"{sys.version_info.major}.{sys.version_info.minor}"
dockerfile = f"""FROM python:{python_version}-slim
WORKDIR /app
COPY requirements.txt ./
RUN python -m pip install -r requirements.txt
COPY run_project.py model.joblib test.csv ./
CMD ["python", "run_project.py"]
"""
(folder / "Dockerfile").write_text(dockerfile, encoding="utf-8")

readme = """# Mock decision-tree evaluation
Invented ECG features, ages and labels; not a medical model.
Start Docker Desktop, then run these commands inside docker_project:

    docker build -t semester-project .
    docker run --rm semester-project

Expected confusion matrix: [[2, 1], [1, 2]].
Rows = actual classes 0, 1; columns = predicted classes 0, 1.
Rebuild after changing project files. Only load model files you trust.
"""
(folder / "README.md").write_text(readme, encoding="utf-8")
print("Ready to build! Project folder:", folder.resolve())


Ready to build! Project folder: docker_project


## 4. Build and run — PowerShell

Open PowerShell at your repository root. Run each line separately:

```powershell
cd docker_project
docker build -t semester-project .
docker run --rm semester-project
```

`cd` enters the folder. `build` packages the project; `-t` names the image and `.` means this folder.
Wait for the build to finish. `run` starts the evaluation; `--rm` removes the container when it finishes.

Expected output:

```text
Confusion matrix (mock test data):
          Predicted 0  Predicted 1
Actual 0            2            1
Actual 1            1            2
```

Four predictions are correct and two are incorrect. With class 1 as positive: TN = 2, FP = 1, FN = 1, TP = 2.
The program prints the table and exits; no browser window is needed.

**Check:** Does Docker match the local result? This checks reproducibility, not performance on real patients.


## 5. Share through GitHub — PowerShell

From `docker_project`, run:

```powershell
cd ..
git add Lecture11_mock_training.ipynb docker_project
git status
git commit -m "Add Docker confusion matrix example"
git push
```

`cd ..` returns to the repository root. `add` selects the files; `status` lets you review them.
`commit` saves a local revision; `push` uploads it to GitHub.

A teammate with repository access opens PowerShell in a new folder and runs the following, replacing `OWNER` and `REPOSITORY`:

```powershell
git clone https://github.com/OWNER/REPOSITORY.git
cd REPOSITORY/docker_project
docker build -t semester-project .
docker run --rm semester-project
```

`clone` downloads the files. Your teammate builds and runs the project without retraining the model or installing Python locally.

For your semester project, use your own evaluated model and a permitted test dataset with known labels.
Only load trusted model files. Keep private patient data out of GitHub.
